# Pseudo-Differential Equation Solvers & Ray Trajectories

This notebook demonstrates numerical simulations for pseudo-differential equations (PDEs), matrix systems, and ray dynamics using the `pde_solver_exponential` package. 

## 1. Imports and Setup

Import necessary scientific libraries along with core solvers, visualization utilities, and trajectory integrators. 

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from psiop import *

## 2. Example A: Scalar 1D Variable-Coefficient Advection-Diffusion

Solves the 1D initial value problem with variable wave speed $c(x) = 0.5 + 0.3\sin(x)$ and viscosity $\nu = 0.02$:

$$\partial_t u = -c(x) \partial_x u + \nu \partial_{xx} u$$

The corresponding symbol is $s(x, \xi) = -i c(x) \xi - \nu \xi^2$. 

In [ ]:
x, xi = sp.symbols('x xi', real=True)
c = 0.5 + 0.3 * sp.sin(x)
nu = 0.02
s_scalar = -sp.I * c * xi - nu * xi**2
f_gauss = lambda X: np.exp(-X**2)

t, U, (xg, kxg) = solve_first_order(
    s_scalar, [x], f_gauss, dt=0.02, n_steps=200, order=3, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)
plot_scalar_1d(t, U, xg, title='Advection-Diffusion', quantity='real')

## 3. Example B: 2x2 Hyperbolic Matrix System

Evolves a coupled 2D vector system $\mathbf{u}(t, x) = [u_1, u_2]^T$ with non-diagonal symbol matrix:

$$S(x, \xi) = \begin{pmatrix} 0 & i\xi \\ i\xi & 0 \end{pmatrix}$$ 

In [ ]:
s_matrix = sp.Matrix([[0, sp.I * xi], [sp.I * xi, 0]])
f_vec = lambda X: [np.exp(-X**2), np.zeros_like(X)]

t2, U2, (xg2, kxg2) = solve_first_order(
    s_matrix, [x], f_vec, dt=0.02, n_steps=200, order=4, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)
plot_matrix_1d(t2, U2, xg2, labels=['u1', 'u2'], quantity='real')

## 4. Example C: Scalar 1D Pure Transport & Animation

Simulates pure linear advection $\partial_t u + 1.5 \partial_x u = 0$ and creates an animated trajectory plot of the field profile over time. 

In [ ]:
c_speed = 1.5
s_transport = -sp.I * c_speed * xi

t3, U3, (xg3, kxg3) = solve_first_order(
    s_transport, [x], f_gauss, dt=0.02, n_steps=200, order=2, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)
anim = animate_scalar_1d(t3, U3, xg3, quantity='real', interval=30)

from IPython.display import HTML
HTML(anim.to_jshtml())

## 5. Example E: 2D Wavepacket Propagation

Evolves a 2D spatial Gaussian wavepacket under the Hamiltonian operator across a 2D numerical spatial grid. 

In [ ]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)
V_hh = (x**2 + y**2)/2 + x**2 * y - y**3/3
H_hh = (xi**2 + eta**2)/2 + V_hh
s_hh = -sp.I * H_hh

def f_wavepacket(X, Y):
    gauss = np.exp(-((X - 0.1)**2)/(2 * 0.5**2) - ((Y - 0.1)**2)/(2 * 0.5**2))
    phase = np.exp(1j * (0.45 * X + 0.35 * Y))
    return gauss * phase

t_hh, U_hh, grids_hh = solve_first_order(
    s_hh, [x, y], f_wavepacket, dt=0.005, n_steps=200, order=2, L=6.0, N=96,
    apply_kwargs=dict(freq_window='gaussian')
)
xg_hh, yg_hh, _, _ = grids_hh
plot_scalar_2d(t_hh, U_hh, xg_hh, yg_hh, quantity='abs')

## 6. Example D: Second-Order PDE (Wave Equation with Potential)

Solves a second-order pseudo-differential equation representing a wave equation with a spatially varying potential:

$$\partial_{tt} u = \partial_{xx} u - V(x)u$$

The corresponding symbol is $s(x, \xi) = -\xi^2 - V(x)$. We use $V(x) = 0.5 \cos(x)$ and initial conditions $u(x,0) = e^{-x^2}$, $\partial_t u(x,0) = 0$.

In [ ]:
x, xi = sp.symbols('x xi', real=True)
V_x = 0.5 * sp.cos(x)
s_wave = -xi**2 - V_x

f_init = lambda X: np.exp(-X**2)
g_init = lambda X: np.zeros_like(X)

t_wave, U_wave, V_wave, (xg_wave, kxg_wave) = solve_second_order(
    s_wave, [x], f_init, g_init, dt=0.05, n_steps=100, order=3, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.pcolormesh(xg_wave, t_wave, np.real(U_wave), shading='auto', cmap='RdBu_r')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('Second-Order PDE: Wave Equation with Potential')
fig.colorbar(im, ax=ax, label='Re(u)')
plt.show()

## 7. Example F: Matrix Field Evolution

Evolves a $2 \times 2$ matrix field $U(t, x)$ under a diagonal matrix symbol representing coupled advection at different speeds.

In [ ]:
S_mat = sp.Matrix([[sp.I * xi, 0], [0, -2 * sp.I * xi]])

def F_mat(X):
    return np.array([
        [np.exp(-X**2), 0.5 * np.exp(-(X-1)**2)],
        [0.5 * np.exp(-(X+1)**2), np.exp(-X**2)]
    ])

t_mf, U_mf, (xg_mf, kxg_mf) = solve_matrix_field(
    S_mat, [x], F_mat, dt=0.05, n_steps=100, order=3, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i in range(2):
    for j in range(2):
        ax = axes[i, j]
        im = ax.pcolormesh(xg_mf, t_mf, np.real(U_mf[:, i, j, :]), shading='auto', cmap='viridis')
        ax.set_title(f'Re(U$_{{{i+1}{j+1}}}$)')
        ax.set_xlabel('x')
        ax.set_ylabel('t')
        fig.colorbar(im, ax=ax)
plt.suptitle('Matrix Field Evolution')
plt.tight_layout()
plt.show()

## 8. Example G: Sylvester Field Evolution

Solves a matrix Sylvester-type equation $\partial_t U = P U + U Q$ using Strang splitting, where $P$ and $Q$ are matrix pseudo-differential operators.

In [ ]:
P_syl = sp.Matrix([[sp.I * xi, 0], [0, 0]])
Q_syl = sp.Matrix([[0, 0], [0, -sp.I * xi]])

def F_syl(X):
    return np.array([
        [np.exp(-X**2), np.exp(-X**2)],
        [np.exp(-X**2), np.exp(-X**2)]
    ])

t_syl, U_syl, (xg_syl, kxg_syl) = solve_sylvester_field(
    P_syl, Q_syl, [x], F_syl, dt=0.05, n_steps=100, order=3, L=10.0, N=256,
    splitting='strang', apply_kwargs=dict(freq_window='gaussian')
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].pcolormesh(xg_syl, t_syl, np.real(U_syl[:, 0, 0, :]), shading='auto', cmap='RdBu_r')
axes[0].set_title('Re(U$_{11}$)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(xg_syl, t_syl, np.real(U_syl[:, 1, 1, :]), shading='auto', cmap='RdBu_r')
axes[1].set_title('Re(U$_{22}$)')
axes[1].set_xlabel('x'); axes[1].set_ylabel('t')
fig.colorbar(im1, ax=axes[1])
plt.suptitle('Sylvester Field Evolution (Strang Splitting)')
plt.tight_layout()
plt.show()

## 9. Example H: 2D Conformal Ricci Flow

Simulates the 2D conformal Ricci flow equation, evolving the conformal factor $\phi(x,y,t)$ of the metric $g = e^{2\phi}(dx^2 + dy^2)$.

In [ ]:
def phi0_2d(X, Y):
    return 0.2 * np.exp(-(X**2 + Y**2)) + 0.1 * np.cos(2*X) * np.cos(2*Y)

t_ricci, phi_ricci, (xg_ricci, yg_ricci) = solve_ricci_flow_conformal_2d(
    phi0_2d, dt=0.005, n_steps=100, order=3, L=4.0, N=64
)

In [ ]:
X_r, Y_r = np.meshgrid(xg_ricci, yg_ricci, indexing='ij')
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
times_to_plot = [0, len(t_ricci)//2, -1]
for idx, t_idx in enumerate(times_to_plot):
    ax = axes[idx]
    im = ax.pcolormesh(X_r, Y_r, phi_ricci[t_idx], shading='auto', cmap='viridis')
    ax.set_title(f't = {t_ricci[t_idx]:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, label=r'$\phi$')
plt.suptitle('2D Conformal Ricci Flow: Conformal Factor $\phi$')
plt.tight_layout()
plt.show()